# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   str    
 1   continent  1704 non-null   str    
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   str    
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 106.6 KB
None
Years: [np.int64(1952), np.int64(1957), np.int64(1962), np.int64(1967), np.int64(1972), np.int64(1977), np.int64(1982), np.int64(1987), np.int64(1992), np.int64(1997), np.int64(2002), np.int64(2007)]
Continents: <StringArray>
['Asia', 'Europe', 'Africa', 'Americas', 'Oceania']
Length: 5, dtype: str
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.

In [4]:
import math

## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [5]:
# Task 1
# YOUR CODE HERE
# Task 1 — Scatter: GDP per capita vs life expectancy
# One continent: Asia
# Two years: 2002 and 2007

asia = df.loc[
    (df["continent"] == "Asia")
    & (df["year"].isin([2002, 2007]))
].copy()

# Convert year to text so Plotly uses two categorical colours
asia["year_label"] = asia["year"].astype(str)

fig1 = px.scatter(
    data_frame=asia,
    x="gdpPercap",
    y="lifeExp",
    color="year_label",
    hover_name="country",
    log_x=True,
    color_discrete_map={
        "2002": "#0072B2",
        "2007": "#D55E00"
    },
    category_orders={"year_label": ["2002", "2007"]},
    labels={
        "gdpPercap": "GDP per Capita (USD, log scale)",
        "lifeExp": "Life Expectancy (years)",
        "year_label": "Year"
    },
    title="Asia’s income and life expectancy changed unevenly, 2002–2007",
    template="plotly_white",
    height=600
)

fig1.update_traces(
    marker=dict(size=9, opacity=0.8)
)

# Connect each country's two observations with a light grey line
# Shapes use the original GDP values, even on a log axis
for country in asia["country"].unique():
    country_data = (
        asia.loc[asia["country"] == country]
        .sort_values("year")
    )

    start = country_data.iloc[0]
    end = country_data.iloc[1]

    fig1.add_shape(
        type="line",
        x0=start["gdpPercap"],
        y0=start["lifeExp"],
        x1=end["gdpPercap"],
        y1=end["lifeExp"],
        line=dict(color="lightgrey", width=1),
        layer="below"
    )

# Highlight India's direction of change
india_2002 = asia.loc[
    (asia["country"] == "India") & (asia["year"] == 2002)
].iloc[0]

india_2007 = asia.loc[
    (asia["country"] == "India") & (asia["year"] == 2007)
].iloc[0]

# Annotation coordinates on a log axis require log10 values
fig1.add_annotation(
    x=math.log10(india_2007["gdpPercap"]),
    y=india_2007["lifeExp"],
    ax=math.log10(india_2002["gdpPercap"]),
    ay=india_2002["lifeExp"],
    xref="x",
    yref="y",
    axref="x",
    ayref="y",
    text="",
    showarrow=True,
    arrowhead=2,
    arrowsize=1.3,
    arrowwidth=2,
    arrowcolor="#D55E00"
)

fig1.add_annotation(
    x=math.log10(india_2007["gdpPercap"]),
    y=india_2007["lifeExp"],
    text="India: higher income and<br>longer life expectancy in 2007",
    showarrow=True,
    arrowhead=1,
    arrowcolor="#D55E00",
    ax=100,
    ay=45,
    font=dict(size=12, family="Arial", color="#D55E00"),
    bgcolor="white",
    bordercolor="#D55E00",
    borderwidth=1,
    borderpad=4
)

fig1.update_layout(
    font=dict(family="Arial", size=13),
    legend=dict(
        orientation="h",
        x=0,
        y=1.06,
        title=""
    ),
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=False),
    margin=dict(t=110)
)

fig1.show()

## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [6]:
# Task 2
# YOUR CODE HERE
# Task 2 — Bubble chart: tell a story
# Full 2007 dataset, with Asia highlighted

df_2007 = df.loc[df["year"] == 2007].copy()

highlight_continent = "Asia"
highlight_colour = "#0072B2"

colour_map = {
    continent: (
        highlight_colour
        if continent == highlight_continent
        else "lightgrey"
    )
    for continent in df_2007["continent"].unique()
}

# Draw Asia last so highlighted bubbles remain visible
continent_order = [
    continent
    for continent in df_2007["continent"].unique()
    if continent != highlight_continent
] + [highlight_continent]

fig2 = px.scatter(
    data_frame=df_2007,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="continent",
    color_discrete_map=colour_map,
    category_orders={"continent": continent_order},
    hover_name="country",
    custom_data=["continent", "pop"],
    log_x=True,
    size_max=60,
    opacity=0.7,
    labels={
        "gdpPercap": "GDP per Capita (USD, log scale)",
        "lifeExp": "Life Expectancy (years)",
        "pop": "Population",
        "continent": "Continent"
    },
    title="Asia spans a wide range of incomes and life expectancies in 2007",
    template="plotly_white",
    height=650
)

# Format the information displayed when hovering
fig2.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Continent: %{customdata[0]}<br>"
        "GDP per capita: $%{x:,.0f}<br>"
        "Life expectancy: %{y:.1f} years<br>"
        "Population: %{customdata[1]:,.0f}"
        "<extra></extra>"
    )
)

# Increase emphasis on Asia
fig2.update_traces(
    selector=dict(name=highlight_continent),
    marker=dict(opacity=0.85)
)

fig2.update_traces(
    selector=lambda trace: trace.name != highlight_continent,
    marker=dict(opacity=0.45)
)

# Annotate two notable countries without labelling every bubble
for country, ax, ay in [
    ("India", 80, 55),
    ("Japan", -100, -45)
]:
    row = df_2007.loc[
        df_2007["country"] == country
    ].iloc[0]

    fig2.add_annotation(
        x=math.log10(row["gdpPercap"]),
        y=row["lifeExp"],
        text=(
            f"<b>{country}</b><br>"
            f"${row['gdpPercap']:,.0f} per person<br>"
            f"{row['lifeExp']:.1f} years"
        ),
        showarrow=True,
        arrowhead=1,
        arrowcolor=highlight_colour,
        ax=ax,
        ay=ay,
        font=dict(
            size=12,
            family="Arial",
            color=highlight_colour
        ),
        bgcolor="white",
        bordercolor=highlight_colour,
        borderwidth=1,
        borderpad=4
    )

# Explain the highlighted group's story
fig2.add_annotation(
    x=0.02,
    y=0.98,
    xref="paper",
    yref="paper",
    xanchor="left",
    yanchor="top",
    text=(
        "<b>Asia highlighted in blue</b><br>"
        "India and Japan illustrate the region’s<br>"
        "large income and life-expectancy differences.<br>"
        "Bubble size represents population."
    ),
    showarrow=False,
    align="left",
    font=dict(
        size=12,
        family="Arial",
        color=highlight_colour
    ),
    bgcolor="white",
    bordercolor=highlight_colour,
    borderwidth=1,
    borderpad=6
)

fig2.update_layout(
    font=dict(family="Arial", size=13),
    showlegend=False,
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=False),
    margin=dict(t=100)
)

fig2.show()